In [1]:
pip install torch transformers sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# -------------------------------
# IMPORTS
# -------------------------------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------------
# 1. LOAD MODEL
# -------------------------------
MODEL_NAME = "microsoft/DialoGPT-medium"  # Lightweight, suitable for console
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
chat_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
chat_model.eval()

print("Chatbot is ready! Type 'exit' to stop.")
print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

# -------------------------------
# 2. INITIALIZE CONVERSATION HISTORY
# -------------------------------
conversation_history = None
MAX_HISTORY_LENGTH = 500  # Limit conversation memory

# -------------------------------
# 3. CHAT LOOP
# -------------------------------
while True:
    user_input = input("User: ").strip()

    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day.")
        break

    # Skip empty input
    if not user_input:
        print("Chatbot: Please enter a valid question.")
        continue

    # -------------------------------
    # Prepare prompt
    # -------------------------------
    prompt_text = f"User: {user_input}\nChatbot:"

    # Tokenization
    encoded_input = tokenizer(prompt_text + tokenizer.eos_token,
                              return_tensors='pt')
    input_ids = encoded_input["input_ids"]
    attention_mask = encoded_input["attention_mask"]

    # Append conversation history if exists
    if conversation_history is not None:
        input_ids = torch.cat([conversation_history, input_ids], dim=-1)
        attention_mask = torch.cat(
            [torch.ones_like(conversation_history), attention_mask], dim=-1
        )

    # Limit memory
    input_ids = input_ids[:, -MAX_HISTORY_LENGTH:]

    # -------------------------------
    # Generate response
    # -------------------------------
    with torch.no_grad():
        conversation_history = chat_model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=input_ids.shape[-1] + 100,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False,
            repetition_penalty=1.2
        )

    # Decode chatbot reply
    bot_reply = tokenizer.decode(
        conversation_history[:, input_ids.shape[-1]:][0],
        skip_special_tokens=True
    ).strip()

    # -------------------------------
    # Display response
    # -------------------------------
    print("Chatbot:", bot_reply)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

🤖 Chatbot is ready! Type 'exit' to stop.
Chatbot: Hello! I am your AI assistant. How can I help you today?



User:  Hello


Chatbot: Hello, I'm a bot. Please message me if you have any questions or concerns!


User:  what is AI


Chatbot: AI analogous to human interaction.


C:\Users\diksh\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\diksh\.cache\huggingface\hub\models--microsoft--DialoGPT-medium. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


User:  who invented python


Chatbot: I think it's more like a conversational language.
